# Accuracy Fix — ImageNet to CIFAR Semantic Mapping

ImageNet models predict 1000 classes. On CIFAR-10 (10 classes) zero-shot accuracy is near 0%.
This notebook maps ImageNet class indices to nearest CIFAR-10 equivalents using WordNet taxonomy,
giving semantically corrected accuracy values for the paper.

**Update DATA_DIR before running.**

In [ ]:
DATA_DIR   = r'E:\cifar-10-python\cifar-10-batches-py'
OUTPUT_DIR = r'E:\decision_consistency_extended'
NUM_SAMPLES=1000; SEED=42; ALPHA=0.5

In [ ]:
import os, pickle, io, csv, random
import numpy as np
import torch, torch.nn.functional as F
import torchvision.transforms.functional as TF
from torchvision import models, transforms
from PIL import Image
from tqdm import tqdm
import warnings; warnings.filterwarnings('ignore')
os.makedirs(OUTPUT_DIR,exist_ok=True)
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic=True
device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:',device)

In [ ]:
# ImageNet -> CIFAR-10 semantic mapping (WordNet taxonomy)
IMAGENET_TO_CIFAR10={
    0:list(range(404,406))+[895],         # airplane
    1:list(range(436,440))+[511,609,627,656,751,817], # automobile
    2:list(range(7,24)),                  # bird
    3:list(range(281,286)),               # cat
    4:[350,351,352,353,354,355],          # deer
    5:list(range(151,269)),               # dog
    6:[30,31,32],                         # frog
    7:[339,340,341,342,603],              # horse
    8:[472,510,536,554,625,628,724,814,833], # ship
    9:[555,569,656,675,717,734,864,867],  # truck
}
imagenet_to_cifar={idx:cifar_label for cifar_label,imagenet_indices in IMAGENET_TO_CIFAR10.items() for idx in imagenet_indices}
def map_imagenet_to_cifar10(pred): return imagenet_to_cifar.get(pred,-1)
print(f'Covered {len(imagenet_to_cifar)} ImageNet classes -> 10 CIFAR classes')

In [ ]:
with open(os.path.join(DATA_DIR,'test_batch'),'rb') as f: d=pickle.load(f,encoding='bytes')
X_test=d[b'data'].reshape(-1,3,32,32); y_test=np.array(d[b'labels'])
X_torch=torch.tensor(X_test,dtype=torch.float32)/255.0; print('Test set:',X_test.shape)

In [ ]:
preprocess=transforms.Compose([transforms.Resize((224,224)),transforms.ToTensor(),transforms.Normalize(mean=[0.485,0.456,0.406],std=[0.229,0.224,0.225])])
def prepare(t): return preprocess(transforms.ToPILImage()(t.cpu().clamp(0,1))).unsqueeze(0).to(device)
def apply_jpeg(t,q=75):
    buf=io.BytesIO(); transforms.ToPILImage()(t.cpu().clamp(0,1)).save(buf,format='JPEG',quality=q); buf.seek(0); return transforms.ToTensor()(Image.open(buf))
def generate_perturbations(t):
    noise=0.01*torch.randn_like(t)
    return {'rotation':TF.rotate(t,angle=2),'brightness':TF.adjust_brightness(t,brightness_factor=1.1),'noise':torch.clamp(t+noise,0,1),'contrast':TF.adjust_contrast(t,contrast_factor=1.1),'blur':TF.gaussian_blur(t,kernel_size=3,sigma=0.5),'jpeg':apply_jpeg(t)}
def infer_model(model,t):
    with torch.no_grad(): probs=F.softmax(model(prepare(t)),dim=1); return probs.argmax(1).item(),probs.max().item()
def compute_metrics(op,oc,pr,alpha=ALPHA):
    K=len(pr); S=sum(1 for p,_ in pr.values() if p==op)/K; D=sum(abs(c-oc)/max(oc,1-oc) for _,c in pr.values())/K
    return {'Label_Stability':S,'Confidence_Deviation':D,'Consistency_Score':alpha*S+(1-alpha)*(1-D)}

In [ ]:
import gc
model_names_list=['ResNet-18','ResNet-50','VGG-16','MobileNetV2']
model_fns={'ResNet-18':models.resnet18,'ResNet-50':models.resnet50,'VGG-16':models.vgg16,'MobileNetV2':models.mobilenet_v2}
results={}; n_eval=min(NUM_SAMPLES,len(y_test))
for mn in model_names_list:
    print(f'\nEvaluating {mn}...')
    model=model_fns[mn](pretrained=True).eval().to(device)
    per_sample=[]
    for idx in tqdm(range(n_eval),desc=mn):
        img=X_torch[idx]; op,oc=infer_model(model,img); mapped_pred=map_imagenet_to_cifar10(op)
        pr={k:infer_model(model,v) for k,v in generate_perturbations(img).items()}
        m=compute_metrics(op,oc,pr); m['true_label']=int(y_test[idx]); m['orig_pred']=op; m['mapped_pred']=mapped_pred; m['correct_mapped']=int(mapped_pred==y_test[idx])
        per_sample.append(m)
    results[mn]=per_sample
    acc=np.mean([m['correct_mapped'] for m in per_sample]); cs=np.mean([m['Consistency_Score'] for m in per_sample])
    print(f'  Acc(mapped)={acc:.3f}  CS={cs:.3f}  CAG={acc-cs:.4f}')
    del model; gc.collect(); torch.cuda.empty_cache()
print('Done.')

In [ ]:
print(f'{"Model":<14} {"Acc(mapped)":>12} {"CS":>7} {"CAG":>8}'); print('-'*46)
rows=[]
for mn in model_names_list:
    ms=results[mn]; acc=np.mean([m['correct_mapped'] for m in ms]); cs=np.mean([m['Consistency_Score'] for m in ms]); cag=acc-cs
    print(f'{mn:<14} {acc:>12.3f} {cs:>7.3f} {cag:>8.4f}'); rows.append({'Model':mn,'Acc_Mapped':round(acc,4),'CS':round(cs,4),'CAG':round(cag,4)})
csv_path=os.path.join(OUTPUT_DIR,'corrected_accuracy_table.csv')
with open(csv_path,'w',newline='') as f: w=csv.DictWriter(f,fieldnames=['Model','Acc_Mapped','CS','CAG']); w.writeheader(); w.writerows(rows)
print('Saved:',csv_path)